In [1]:
import pandas as pd

df=pd.read_csv("250k_rndm_zinc_drugs_clean_3.csv")

df.shape

(249455, 4)

In [2]:
df.head(5)

,smiles,logP,qed,SAS
0,CC(C)(C)c1ccc2occ(CC(=O)Nc3ccccc3F)c2c1\n,5.05060,0.702012,2.084095
1,C[C@@H]1CC(Nc2cncc(-c3nncn3C)c2)C[C@@H](C)C1\n,3.11370,0.928975,3.432004
2,N#Cc1ccc(-c2ccc(O[C@@H](C(=O)N3CCCC3)c3ccccc3)...,4.96778,0.599682,2.470633
3,CCOC(=O)[C@@H]1CCCN(C(=O)c2nc(-c3ccc(C)cc3)n3c...,4.00022,0.690944,2.822753
4,N#CC1=C(SCC(=O)Nc2cccc(Cl)c2)N=C([O-])[C@H](C#...,3.60956,0.789027,4.035182


In [3]:
print(df["qed"].mean())

0.7282644882810482


In [4]:
print(df["logP"].mean())

2.4570930029063356


In [5]:
def calculate_average_molecular_weight(df, smiles_column='smiles'):
    """
    Calculate the average molecular weight of compounds in a DataFrame.
    
    Args:
        df (pandas.DataFrame): DataFrame containing SMILES strings
        smiles_column (str): Name of the column containing SMILES strings
        
    Returns:
        tuple: (float, pd.Series) - Average molecular weight and Series of all weights
    """
    from rdkit import Chem
    from rdkit.Chem import Descriptors
    import numpy as np
    from tqdm import tqdm
    
    molecular_weights = []
    valid_count = 0
    
    print(f"Calculating molecular weights for {len(df)} compounds...")
    
    # Calculate molecular weight for each valid molecule
    for _, row in tqdm(df.iterrows(), total=len(df)):
        try:
            smiles = row[smiles_column]
            mol = Chem.MolFromSmiles(smiles)
            
            if mol:
                mw = Descriptors.MolWt(mol)
                molecular_weights.append(mw)
                valid_count += 1
        except Exception as e:
            continue
    
    # Convert to numpy array for calculations
    mw_array = np.array(molecular_weights)
    
    # Calculate statistics
    average_mw = np.mean(mw_array)
    median_mw = np.median(mw_array)
    min_mw = np.min(mw_array)
    max_mw = np.max(mw_array)
    std_mw = np.std(mw_array)
    
    # Print summary
    print(f"\nMolecular Weight Statistics:")
    print(f"  Processed {valid_count} valid molecules out of {len(df)}")
    print(f"  Average MW: {average_mw:.2f}")
    print(f"  Median MW: {median_mw:.2f}")
    print(f"  Range: {min_mw:.2f} - {max_mw:.2f}")
    print(f"  Standard Deviation: {std_mw:.2f}")
    
    # Distribution in common ranges
    ranges = [(0, 200), (200, 350), (350, 500), (500, 800), (800, float('inf'))]
    range_labels = ['< 200', '200-350', '350-500', '500-800', '> 800']
    
    print("\nMolecular Weight Distribution:")
    for (lower, upper), label in zip(ranges, range_labels):
        count = np.sum((mw_array >= lower) & (mw_array < upper))
        percentage = (count / len(mw_array)) * 100
        print(f"  {label}: {count} molecules ({percentage:.1f}%)")
    
    return average_mw, mw_array


import pandas as pd
    
# Example with a dataframe
df = pd.read_csv('250k_rndm_zinc_drugs_clean_3.csv')
avg_mw, all_mws = calculate_average_molecular_weight(df, 'smiles')
    
# You can also save the results
df_with_mw = df.copy()
df_with_mw['molecular_weight'] = pd.Series(all_mws)
df_with_mw.to_csv('zinc_with_mw.csv', index=False)

Calculating molecular weights for 249455 compounds...


100%|██████████████████████████████████████████████████████████████████| 249455/249455 [01:06<00:00, 3728.08it/s]



Molecular Weight Statistics:
  Processed 249455 valid molecules out of 249455
  Average MW: 332.14
  Median MW: 333.82
  Range: 150.12 - 500.00
  Standard Deviation: 61.94

Molecular Weight Distribution:
  < 200: 4201 molecules (1.7%)
  200-350: 150137 molecules (60.2%)
  350-500: 95116 molecules (38.1%)
  500-800: 1 molecules (0.0%)
  > 800: 0 molecules (0.0%)
